# Fine-tune CodeBERT — LAMPS

Files lấy từ Google Drive tại `/content/drive/My Drive/NT230/`

| File | Path trên Drive |
|---|---|
| `train.jsonl` | `NT230/train.jsonl` |
| `val.jsonl` | `NT230/val.jsonl` |
| `test.jsonl` | `NT230/test.jsonl` |
| `run.py` | `NT230/run.py` |
| `model.py` | `NT230/model.py` |

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

DRIVE_DIR = '/content/drive/My Drive/NT230'
WORK_DIR  = '/content'

# Copy files từ Drive về /content để run.py chạy được
required = ['train.jsonl', 'val.jsonl', 'test.jsonl', 'run.py', 'model.py']
for f in required:
    src = f'{DRIVE_DIR}/{f}'
    dst = f'{WORK_DIR}/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✅ Copied: {f}')
    else:
        print(f'❌ MISSING on Drive: {src}')

In [ ]:
!pip install -q transformers==4.40.0 torch accelerate tqdm

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ No GPU — Runtime → Change runtime type → T4')

In [ ]:
# Fine-tune CodeBERT
!python /content/run.py \
  --output_dir=/content/saved_models \
  --model_type=roberta \
  --tokenizer_name=microsoft/codebert-base \
  --model_name_or_path=microsoft/codebert-base \
  --do_train --do_eval --do_test \
  --train_data_file=/content/train.jsonl \
  --eval_data_file=/content/val.jsonl \
  --test_data_file=/content/test.jsonl \
  --epoch 4 \
  --block_size 400 \
  --train_batch_size 16 \
  --eval_batch_size 64 \
  --learning_rate 2e-5 \
  --max_grad_norm 1.0 \
  --evaluate_during_training \
  --seed 123456

In [ ]:
# Kiểm tra checkpoint
ckpt = '/content/saved_models/checkpoint-best-acc/model.bin'
if os.path.exists(ckpt):
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'✅ Checkpoint: {ckpt}  ({size_mb:.0f} MB)')
else:
    print('❌ Checkpoint không tìm thấy — xem log training ở trên')

In [ ]:
# Lưu checkpoint về Drive
import shutil
src = '/content/saved_models'
dst = f'{DRIVE_DIR}/saved_models'
shutil.copytree(src, dst, dirs_exist_ok=True)
print(f'✅ Saved to Drive: {dst}')
print('Tải về: model.bin nằm tại NT230/saved_models/checkpoint-best-acc/model.bin')